# 0장 실습. 실습 환경 점검

이 책의 노트북은 모두 `jupyter-new/` 에서 실행한다. 노트북 하나가 필요한 코드를 모두 담고 있어서, 어느 노트북이든 혼자 열어 처음부터 끝까지 돌릴 수 있다. 노트북끼리는 파일만 주고받는다.

| 폴더 | 들어가는 것 |
|---|---|
| `data/raw/` | 공개 원본 데이터 (이 노트북이 받는다) |
| `data/` | 노트북이 만든 데이터셋 (8장의 평가셋 등) |
| `results/` | 엔진별 평가 기록 |
| `models/` | 파인튜닝한 체크포인트 |
| `out/` | 그림과 로그 |

이 노트북은 실습 전에 한 번 돌린다. 다섯 가지를 확인한다.

1. 파이썬 패키지와 버전
2. GPU
3. `.env` 의 설정
4. 공개 원본 데이터
5. 엔진 둘 — Jev API 와 사내 GPU 의 로컬 LLM

로컬 LLM 은 서빙 엔진 없이 노트북 안에서 transformers 로 직접 올린다. 처음 실행하면 Hugging Face 에서 모델을 받는다(Qwen3-8B 는 약 16GB).

**`.env`.** jupyter notebook 경로에 아래를 적는다. 이 파일은 공유하지 않는다.

```
TYPESAFE_API_KEY=발급받은 키
LOCAL_LLM_MODEL=Qwen/Qwen3-8B
```

In [8]:
import os, time
from pathlib import Path
from importlib.metadata import version
import pandas as pd, requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter, Retry

load_dotenv()
DATA, RAW = Path("data"), Path("data/raw")
for d in [RAW, Path("results"), Path("models"), Path("out")]:
    d.mkdir(parents=True, exist_ok=True)

http = requests.Session()
retry = HTTPAdapter(max_retries=Retry(total=4, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504]))
http.mount("https://", retry)

## 0.1 패키지와 버전

책의 숫자는 이 버전에서 나왔다. 설치되지 않은 패키지가 있으면 이 셀이 `PackageNotFoundError` 로 멈춘다. 그 이름을 `pip install` 한다.

In [9]:
%pip install -q accelerate

Note: you may need to restart the kernel to use updated packages.


In [10]:
PACKAGES = ["numpy", "pandas", "matplotlib", "koreanize-matplotlib", "scikit-learn", "requests", "python-dotenv",
            "torch", "transformers", "accelerate", "laya", "langgraph"]
pd.Series({p: version(p) for p in PACKAGES}, name="버전").to_frame()

,버전
numpy,2.4.6
pandas,3.0.6
matplotlib,3.11.2
koreanize-matplotlib,0.1.1
scikit-learn,1.9.1
requests,2.34.2
python-dotenv,1.2.4
torch,2.12.0+cu132
transformers,5.19.0
accelerate,1.15.0


## 0.2 GPU

로컬 LLM 과 Laya 가 GPU 에서 돈다. Qwen3-8B 는 bf16 으로 약 16GB, Laya 파인튜닝은 10GB 안팎을 쓴다. 한 노트북 안에서 둘을 동시에 올리지 않는다.

In [11]:
import torch

free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f"| GPU {torch.cuda.device_count()}장 | 메모리 {total / 2**30:.1f}GB 중 {free / 2**30:.1f}GB 비어 있음")

NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition | GPU 2장 | 메모리 95.0GB 중 46.5GB 비어 있음


## 0.3 설정 확인

`.env` 를 읽었는지 본다. 키는 앞뒤 네 글자만 보여 준다.

In [12]:
mask = lambda v: v[:4] + "…" + v[-4:]
pd.Series({"TYPESAFE_API_KEY": mask(os.environ["TYPESAFE_API_KEY"]),
           "LOCAL_LLM_MODEL": os.environ["LOCAL_LLM_MODEL"]}, name="값").to_frame()

,값
TYPESAFE_API_KEY,apik…e9e9
LOCAL_LLM_MODEL,Qwen/Qwen3.5-4B


## 0.4 공개 원본 데이터

원본은 책 저장소에 넣지 않는다. 공개 저장소에서 받는다. 이미 받은 파일은 건너뛴다.

| 파일 | 원본 | 라이선스 | 쓰는 장 |
|---|---|---|---|
| ynat | KLUE 뉴스 제목 분류 | CC BY-SA 4.0 | 8장 평가셋, 4부 |
| klue-nli | KLUE 자연어 추론 | CC BY-SA 4.0 | 8장 평가셋, 4부 |
| klue-sts | KLUE 문장 유사도 | CC BY-SA 4.0 | 8장 평가셋, 4부 |
| klue-mrc | KLUE 기계 독해 | CC BY-SA 4.0 | 6장 재순위, 19장 RAG 라우터 |
| ratings | NSMC 영화 리뷰 감정 | CC0 1.0 | 8장 평가셋, 4부 |
| deepset/prompt-injections | 영어·독일어 프롬프트 인젝션 (Hugging Face, 18장이 직접 받는다) | Apache-2.0 | 18장 증류 |

In [13]:
KLUE = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark"
NSMC = "https://raw.githubusercontent.com/e9t/nsmc/master"
FILES = {f"{name}_{split}.json": f"{KLUE}/{name}/{name}_{split}.json"
         for name in ["ynat-v1.1", "klue-nli-v1.1", "klue-sts-v1.1", "klue-mrc-v1.1"] for split in ["train", "dev"]}
FILES |= {f"ratings_{split}.txt": f"{NSMC}/ratings_{split}.txt" for split in ["train", "test"]}

for name in [n for n in FILES if not (RAW / n).exists()]:
    (RAW / name).write_bytes(http.get(FILES[name], timeout=300).content)
pd.Series({n: f"{(RAW / n).stat().st_size / 2**20:.1f} MB" for n in FILES}, name="크기").to_frame()

,크기
ynat-v1.1_train.json,44.0 MB
ynat-v1.1_dev.json,8.8 MB
klue-nli-v1.1_train.json,12.0 MB
klue-nli-v1.1_dev.json,1.4 MB
klue-sts-v1.1_train.json,9.8 MB
klue-sts-v1.1_dev.json,0.4 MB
klue-mrc-v1.1_train.json,45.7 MB
klue-mrc-v1.1_dev.json,17.8 MB
ratings_train.txt,14.0 MB
ratings_test.txt,4.7 MB


## 0.5 Jev API

판단 하나를 보내 키와 연결을 확인한다. 응답의 `model` 이 지금 `jev-latest` 가 가리키는 버전이다.

In [14]:
JEV_URL = os.getenv("JEV_URL", "https://api.typesafe.ai")
r = http.post(f"{JEV_URL}/v1/systemone", timeout=60,
              headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
              json={"model": "jev-latest", "state": "환불은 언제 되나요?",
                    "questions": {"refund": {"type": "noul", "instructions": "고객이 환불에 대해 묻는다"}}})
r.raise_for_status()
print(r.json()["model"], r.json()["answers"], f"| {r.elapsed.total_seconds() * 1000:.0f} ms")

jev-1.13.0 {'refund': {'type': 'noul', 'noul': 0.98}} | 213 ms


## 0.6 로컬 LLM

모델을 올리고 한 문장을 물어 한국어 응답과 지연을 확인한다. 이 `chat()` 이 로컬 LLM 을 쓰는 장마다 그대로 들어간다. Qwen3 계열은 기본으로 생각 과정을 길게 쓰므로 `enable_thinking=False` 로 답만 받는다. 이 옵션은 그런 설정이 없는 모델의 채팅 템플릿에서는 무시된다.

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = os.environ["LOCAL_LLM_MODEL"]
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(LLM_MODEL, padding_side="left")
lm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype="auto", device_map="auto")
print(f"{LLM_MODEL} 올림 | {time.perf_counter() - t0:.0f}초 | {lm.device} | 파라미터 {lm.num_parameters() / 1e9:.1f}B")

def chat(prompt, max_new_tokens=256):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)
    enc = tok(text, return_tensors="pt").to(lm.device)
    out = lm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

t0 = time.perf_counter()
print(chat("결정 모델과 생성 모델의 차이를 한 문장으로 말해 줘.", max_new_tokens=80), f"\n| {time.perf_counter() - t0:.2f}초")

/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring corrupted tree cache file /data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json: [Errno 13] Permission denied: '/data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json'
Ignoring corrupted tree cache file /data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json: [Errno 13] Permission denied: '/data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json'
Ignoring corrupted tree cache file /data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json: [Errno 13] Permission denied: '/data/shared/hf/hub/models--Qwen--Qwen3.5-4B

Qwen/Qwen3.5-4B 올림 | 4초 | cuda:0 | 파라미터 4.2B


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


결정 모델은 주어진 입력에 대해 정답을 예측하거나 분류하는 것을 목표로 하는 반면, 생성 모델은 새로운 데이터나 콘텐츠를 만들어내는 것을 목표로 합니다.
 
| 2.10초


In [16]:
# 다음 노트북을 위해 GPU 메모리를 비운다
del lm
torch.cuda.empty_cache()

## 정리

- 패키지 버전, GPU, `.env`, 원본 데이터, Jev, 로컬 LLM 을 확인했다. 로컬 LLM 은 처음 실행에서 받아 두었으므로 뒤의 장에서는 바로 올라간다.
- 출력된 버전과 모델 이름을 책머리의 기준 버전과 견준다. 다르면 숫자가 조금씩 달라질 수 있다.